# 🌳 Create Main Comparison Function

**날짜**: 2026-05-25  
**목표**: 함수 3개를 통합한 메인 함수 + .py 모듈 만들기

## 🛠️ 만들 함수
- `compare_before_after()`: 산불 전후 비교 메인 함수
- `ndvi_analyzer.py`: 재사용 가능한 파이썬 모듈

## 🔎 분석 방법 (Day 6에서 확정)
- 산불 전 = 작년 5월 (2024-05)
- 산불 후 = 올해 5월 (2025-05)
- → 같은 계절 비교로 계절 효과 제거

In [2]:
# 환경
import ee
import geemap
ee.Initialize(project='knu-project-ndvi-2026')

In [3]:
# 1st, 2nd에서 만든 함수 3개
def get_sentinel2_image(area, start_date, end_date, cloud_threshold=10):
    """Sentinel-2 영상 1장 가져오기"""
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(area)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
            .first())

def calculate_ndvi(image):
    """NDVI 계산"""
    return image.normalizedDifference(['B8', 'B4']).rename('NDVI')

def get_mean_ndvi(ndvi_image, area):
    """평균 NDVI 추출"""
    mean_dict = ndvi_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area,
        scale=10,
        maxPixels=1e9
    )
    return mean_dict.get('NDVI').getInfo()

print("✅ 함수 3개 준비 완료!")

✅ 함수 3개 준비 완료!


In [4]:
def compare_before_after(area, period_before, period_after, cloud_threshold=10):
    """
    산불 전후 NDVI를 비교 분석한다.
    
    매개변수:
        area (ee.Geometry): 분석 영역
        period_before (dict): 산불 전 시기 {'start': '...', 'end': '...'}
        period_after (dict): 산불 후 시기 {'start': '...', 'end': '...'}
        cloud_threshold (int): 구름 비율 한계 (기본 10)
    
    반환값:
        dict: 비교 결과 (NDVI 값, 영상, 날짜, 변화율 모두 포함)
    """
    # 산불 전 처리
    img_before = get_sentinel2_image(area, period_before['start'], period_before['end'], cloud_threshold)
    ndvi_before = calculate_ndvi(img_before)
    mean_before = get_mean_ndvi(ndvi_before, area)
    date_before = ee.Date(img_before.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    
    # 산불 후 처리
    img_after = get_sentinel2_image(area, period_after['start'], period_after['end'], cloud_threshold)
    ndvi_after = calculate_ndvi(img_after)
    mean_after = get_mean_ndvi(ndvi_after, area)
    date_after = ee.Date(img_after.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    
    # 변화 계산
    diff = mean_after - mean_before
    percent_change = (diff / mean_before) * 100
    
    # 결과를 딕셔너리로 묶어서 반환
    return {
        'mean_before': mean_before,
        'mean_after': mean_after,
        'date_before': date_before,
        'date_after': date_after,
        'diff': diff,
        'percent_change': percent_change,
        'ndvi_before_image': ndvi_before,
        'ndvi_after_image': ndvi_after
    }

print("✅ compare_before_after() 함수 정의 완료!")

✅ compare_before_after() 함수 정의 완료!


In [6]:
# 분석 영역 (확정한 좌표)
area = ee.Geometry.Point([128.5774, 35.9145]).buffer(800)

# 시기 (작년 5월 vs 올해 5월)
period_before = {'start': '2024-05-01', 'end': '2024-05-31'}
period_after = {'start': '2025-05-01', 'end': '2025-05-31'}

# 한 줄로 전체 분석!
result = compare_before_after(area, period_before, period_after)

# 결과 출력
print("=" * 40)
print("🔥 함지산 산불 NDVI 분석 결과")
print("=" * 40)
print(f"기준 시점 (작년): {result['date_before']}")
print(f"산불 후 (올해): {result['date_after']}")
print()
print(f"산불 전 NDVI: {result['mean_before']:.3f}")
print(f"산불 후 NDVI: {result['mean_after']:.3f}")
print(f"변화율: {result['percent_change']:.1f}%")
print("=" * 40)

🔥 함지산 산불 NDVI 분석 결과
기준 시점 (작년): 2024-05-03
산불 후 (올해): 2025-05-08

산불 전 NDVI: 0.693
산불 후 NDVI: 0.484
변화율: -30.2%
